# 01 — Data Collection

## Objective

Acquire a Country x Year panel of macroeconomic, consumer-spending, demographic, and digitalisation indicators covering as many countries as reliable data allows, from an authoritative Tier-1 source.

## Source Selection

Three Tier-1 sources were evaluated for the core panel: **World Bank** (Open Data API), **IMF** (World Economic Outlook database), and **OECD** (SDMX API).

| Source | Country coverage | Consumer-spending indicator | API access | Decision |
|---|---|---|---|---|
| **World Bank** | 217 economies | Household Final Consumption Expenditure (NE.CON.PRVT.*) — direct, annual, 1960s–present | Simple REST, no key required, JSON | **Selected** — best combination of coverage, a genuine consumer-spending variable, and ease of reproducible access |
| IMF WEO | ~195 countries | No direct household consumption expenditure series in the core WEO database (GDP components available only for a subset) | Requires SDMX querying, more complex | Not selected as primary; consistent with World Bank being sufficient |
| OECD | 38 members + partners | Full COICOP category-level detail available | SDMX API, complex dimension keys | Evaluated for category-level spending (Section 10 of brief); **v1 descoped it** (dimension keys could not be resolved), **v2 resolved it** by discovering the real keys via the API's own `availableconstraint` endpoint rather than guessing — 36 countries of real category-level data now used as a supplementary layer (`src/data_collection/oecd_categories.py`, `docs/LIMITATIONS.md` item 1) |

**Decision: World Bank Open Data is the primary and sole quantitative source for the core Country x Year panel.** Using one consistent source for every indicator avoids the definitional and methodological mismatches that plague multi-source international panels (a country's GDP measured by two different agencies is rarely identical) — directly addressing Section 8 and Section 18's warnings about comparability. Two supplementary sources were layered on top in v2 without disturbing this core panel: World Bank Worldwide Governance Indicators (for the Market Attractiveness Index's Market Stability pillar) and OECD category-level spending (above).

## Indicators Collected

14 core World Bank indicators, 2013–2023, all 217 non-aggregate economies: GDP (current USD), GDP per capita (current USD and PPP), population, household consumption expenditure (current USD, constant 2015 USD, per capita, % of GDP), inflation (CPI), unemployment, urban population %, internet users %, population 65+, age dependency ratio. Two supplementary Global Findex indicators (account ownership, digitally-enabled account) were also collected as periodic cross-sectional snapshots. **v2 adds** 3 World Bank Worldwide Governance Indicators (political stability, rule of law, regulatory quality) and OECD's 12-category COICOP household spending breakdown for 36 countries.

Full indicator definitions: [`docs/DATA_DICTIONARY.md`](../docs/DATA_DICTIONARY.md). Full source citations: [`docs/SOURCES.md`](../docs/SOURCES.md).

## Collection Method

`src/data_collection/world_bank.py` calls `api.worldbank.org/v2/country/all/indicator/{code}` once per indicator, paginating through all results, and writes both per-indicator CSVs (`data/raw/worldbank/`) and a combined wide panel (`data/raw/worldbank_panel_wide.csv`). No API key is required and the World Bank API terms permit redistribution (CC-BY 4.0), so the raw pulled data is committed to this repository for full reproducibility. `src/data_collection/governance_indicators.py` (v2) follows the identical pattern for the 3 WGI indicators. `src/data_collection/oecd_categories.py` (v2) queries the OECD SDMX API instead — see that script's docstring for the exact dimension key and how it was discovered.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src/visualisation')
panel = pd.read_csv('../data/raw/worldbank_panel_wide.csv')
print('Raw panel shape (includes WB aggregates, filtered out later):', panel.shape)
panel.head()

Raw panel shape (includes WB aggregates, filtered out later): (2915, 17)


,iso3,year,country_name,gdp_current_usd,gdp_per_capita_current_usd,gdp_per_capita_ppp_current_intl,population_total,household_consumption_expenditure_current_usd,household_consumption_expenditure_constant_2015_usd,household_consumption_expenditure_per_capita_constant_2015_usd,household_consumption_pct_of_gdp,inflation_cpi_annual_pct,unemployment_pct_of_labor_force,urban_population_pct,internet_users_pct_of_population,population_65_plus_pct,age_dependency_ratio_pct
0,AFE,2023,Africa Eastern and Southern,1.179122e+12,1571.132704,4496.744898,750491370.0,7.957335e+11,7.549310e+11,1005.915583,67.485282,7.399186,7.676548,37.772301,27.8,3.286303,79.147520
1,AFE,2022,Africa Eastern and Southern,1.226461e+12,1675.902524,4359.591407,731821393.0,8.074927e+11,7.353041e+11,1004.758994,65.839233,10.883478,7.869470,37.360578,26.8,3.246111,79.879463
2,AFE,2021,Africa Eastern and Southern,1.113060e+12,1560.894626,4022.277659,713090928.0,7.263121e+11,7.055961e+11,989.489654,65.253642,6.824727,8.407412,36.908543,25.0,3.216573,80.634338
3,AFE,2020,Africa Eastern and Southern,9.385461e+11,1351.503167,3707.033612,694446100.0,6.257222e+11,6.659466e+11,958.960804,66.669310,5.191629,7.974302,36.488322,23.5,3.192169,81.433777
4,AFE,2019,Africa Eastern and Southern,1.019354e+12,1508.031525,3804.533834,675950189.0,6.783635e+11,6.726972e+11,995.187569,66.548356,4.102851,7.459106,36.097331,21.6,3.157142,82.236098


## World Bank's own country/region metadata (used to exclude aggregates and build the region mapping)

In [2]:
import json
wb_meta = json.load(open('../data/raw/wb_countries.json', encoding='utf-8'))[1]
wb_df = pd.json_normalize(wb_meta)
print('Total entities returned by /v2/country:', len(wb_df))
print('Of which real countries (region != Aggregates):', (wb_df['region.value']!='Aggregates').sum())
wb_df[wb_df['region.value']!='Aggregates'][['id','name','region.value','incomeLevel.value']].head()

Total entities returned by /v2/country: 295
Of which real countries (region != Aggregates): 217


,id,name,region.value,incomeLevel.value
0,ABW,Aruba,Latin America & Caribbean,High income
2,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
5,AGO,Angola,Sub-Saharan Africa,Lower middle income
6,ALB,Albania,Europe & Central Asia,Upper middle income
7,AND,Andorra,Europe & Central Asia,High income


## Data Collection Outcome

- 217 real economies returned by the World Bank country list, of which **182 (84%)** have at least one year of consumer-spending (household consumption expenditure) data in the 2013–2023 window — see Notebook 02 for the completeness breakdown by indicator.
- This comfortably exceeds the 100+ country target while using only real, PUBLIC, directly-fetched data — no country was included with a fabricated or estimated figure to hit a coverage number.